[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/089.%20The%20Single%20vs.%20Dual%20Sourcing%20Problem/P89-Tier-5_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 89. The Single vs. Dual Sourcing Problem

## Tier 5: An Integrated Digital-Twin Simulation

### Purpose and scope

This notebook builds a small, synthetic digital-twin-style simulation for sourcing decisions. It connects simulated sensor observations, supplier and market metrics, simple predictive models, rule-based sourcing recommendations, alerts, and what-if scenarios. The example uses three components and four suppliers.

The simulation illustrates how these pieces can be organized and connected; it is not a deployed digital twin. It does not connect to live ERP, supply-chain, IoT, or market systems. Its forecasts and performance metrics are illustrative rather than empirically trained or validated. The simulation generates one observation per configured sensor per simulated minute; the sensor sampling-rate fields describe configuration metadata and do not drive sub-minute sampling.

### Inputs and outputs

- **Inputs:** component and supplier records, synthetic sensor observations, and scenario parameters.
- **Outputs:** simulated performance histories, single- or dual-sourcing recommendations, alerts, summary metrics, scenario summaries, and visualizations.
- **Decision rule:** a supplier score combines predicted risk, predicted cost, and the supplier's reliability score. Components with a criticality score greater than 0.8 are assigned to the two highest-scoring suppliers; other components are assigned to the highest-scoring supplier.

The recommendations are rules, not solutions to an optimization model. In particular, this notebook does not optimize allocation ratios, enforce supplier capacity limits, or calculate a cost-minimizing sourcing plan. Dual sourcing uses a fixed 70%/30% allocation for illustration.

### Simulation sequence

1. Define the records that represent sensor observations, supplier performance, market conditions, demand forecasts, decisions, and system metrics.
2. Generate synthetic sensor observations and update the simulated operational state.
3. Apply the simple forecasting and risk/cost prediction models.
4. Rank suppliers and make sourcing recommendations using the stated decision rule.
5. Check alerts, summarize system metrics, and explore what-if scenarios.

### Modeling assumptions

- Sensor values and several operational metrics are generated from probability distributions, so results vary between runs. No random seed is set.
- The simulated demand forecast uses a fixed growth factor and random noise; it is not a fitted time-series model.
- The risk and cost models are transparent illustrative scoring rules, not calibrated predictive models.
- Scenario runs start from a fresh simulated history and restore the base simulation afterward. Scenario results therefore describe the scenario run, not a continuation of the base run.
- Supplier capacity, order quantities, lead-time feasibility, contractual terms, and component-specific supplier eligibility are not modeled.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Representing the simulated system

A data class is a compact way to define a record with named fields. These records make the simulation's data flow explicit: sensor readings become performance and demand records, which in turn inform recommendations and summary metrics. Scores such as reliability and risk are represented on a 0–1 scale unless a field explicitly states another unit.

In [ ]:
@dataclass
class IoTDataPoint:
    """One synthetic sensor observation."""

    sensor_id: str
    timestamp: datetime
    value: float
    unit: str
    location: str
    quality_score: float


@dataclass
class SupplierPerformance:
    """Supplier performance values for one simulated time step."""

    supplier_id: str
    timestamp: datetime
    on_time_delivery_rate: float
    quality_score: float
    cost_performance: float
    inventory_turnover: float
    disruption_risk_score: float
    capacity_utilization: float
    response_time: float


@dataclass
class MarketCondition:
    """Synthetic market indicators for one simulated time step."""

    timestamp: datetime
    market_index: float
    commodity_price_index: float
    shipping_cost_index: float
    currency_exchange_rate: float
    geopolitical_risk_score: float
    demand_volatility: float


@dataclass
class ComponentDemand:
    """Current demand and illustrative forecast for one component."""

    component_id: str
    timestamp: datetime
    current_demand: float
    forecast_7_days: float
    forecast_30_days: float
    forecast_confidence: float
    safety_stock_level: float
    reorder_point: float


@dataclass
class SourcingDecision:
    """A rule-based sourcing recommendation and its illustrative impacts."""

    decision_id: str
    timestamp: datetime
    component_id: str
    action_type: str
    selected_suppliers: List[str]
    allocation_ratios: List[float]
    expected_cost_impact: float
    expected_service_impact: float
    expected_risk_impact: float
    confidence_score: float
    implementation_time: timedelta


@dataclass
class DigitalTwinMetrics:
    """Simulated data, prediction, and system-performance metrics."""

    timestamp: datetime
    data_quality_score: float
    prediction_accuracy: float
    system_uptime: float
    response_time_ms: float
    synchronization_delay_ms: float
    active_sensors: int
    data_points_processed: int
    alerts_generated: int
    decisions_supported: int

## 2. Define prediction and support components

The prediction classes below are deliberately simple so that their assumptions can be inspected. `DemandForecastModel` scales current demand by a horizon-dependent growth factor and adds noise. `SupplierRiskModel` combines a performance-based score with a market-based score. `CostPredictionModel` adjusts the supplier's given unit cost using the latest commodity, shipping, and exchange-rate indicators. These models illustrate interfaces and data flow; their outputs should not be interpreted as validated forecasts.

In [ ]:
class DemandForecastModel:
    """Illustrative demand forecast using a fixed growth factor and noise."""

    def predict(self, component_id: str, current_demand: float, days_ahead: int) -> float:
        alpha = 0.3  # Retained as a model configuration parameter for this example.
        trend_factor = 1.0 + days_ahead * 0.01
        noise = np.random.normal(0.0, 0.05)
        predicted_demand = current_demand * trend_factor * (1.0 + noise)
        return max(0.0, predicted_demand)


class SupplierRiskModel:
    """Combine performance and market indicators into a bounded risk score."""

    def predict(
        self,
        performance: SupplierPerformance,
        market_conditions: List[MarketCondition],
    ) -> float:
        performance_risk = 1.0 - (
            performance.on_time_delivery_rate * 0.4
            + performance.quality_score * 0.3
            + (1.0 - performance.capacity_utilization) * 0.3
        )

        market_risk = 0.0
        if market_conditions:
            latest_market = market_conditions[-1]
            market_risk = (
                latest_market.geopolitical_risk_score * 0.5
                + abs(latest_market.market_index) * 0.3
                + latest_market.demand_volatility * 0.2
            )

        total_risk = performance_risk * 0.7 + market_risk * 0.3
        return float(np.clip(total_risk, 0.0, 1.0))


class CostPredictionModel:
    """Apply illustrative market adjustments to a supplier's recorded unit cost."""

    def predict(
        self,
        supplier_id: str,
        base_cost: float,
        market_conditions: List[MarketCondition],
    ) -> float:
        market_adjustment = 1.0
        if market_conditions:
            latest_market = market_conditions[-1]
            market_adjustment = (
                1.0
                + latest_market.commodity_price_index / 1000.0
                + latest_market.shipping_cost_index / 1000.0
                + latest_market.currency_exchange_rate
                - 1.0
            )

        return max(0.0, base_cost * market_adjustment)


class DisruptionPredictionModel:
    """Estimate an illustrative system-wide disruption-risk score."""

    def predict(
        self,
        market_conditions: List[MarketCondition],
        suppliers: List[Dict[str, Any]],
    ) -> float:
        base_risk = 0.1
        market_impact = 0.0
        if market_conditions:
            latest_market = market_conditions[-1]
            market_impact = (
                latest_market.geopolitical_risk_score * 0.6
                + abs(latest_market.market_index) * 0.4
            )

        concentration_risk = 0.1 * (1.0 - len(suppliers) / 10.0)
        return float(np.clip(base_risk + market_impact + concentration_risk, 0.0, 1.0))


class ScenarioEngine:
    """Provide named what-if cases and their input parameters."""

    def __init__(self) -> None:
        self.scenarios = {
            "supplier_disruption": {
                "description": "Disruption affecting one supplier",
                "params": {"supplier_disruption": "supplier_1"},
                "duration_minutes": 30,
            },
            "demand_surge": {
                "description": "Increase in component demand forecasts",
                "params": {"demand_surge": 1.5},
                "duration_minutes": 30,
            },
            "cost_increase": {
                "description": "Increase in supplier unit costs",
                "params": {"cost_increase": 1.2},
                "duration_minutes": 30,
            },
            "combined_crisis": {
                "description": "Combined supplier, demand, and cost pressures",
                "params": {
                    "supplier_disruption": "supplier_1",
                    "demand_surge": 1.3,
                    "cost_increase": 1.15,
                },
                "duration_minutes": 45,
            },
        }

    def get_available_scenarios(self) -> Dict[str, Dict[str, Any]]:
        return self.scenarios


class AlertSystem:
    """Store and categorize alerts generated by the simulation."""

    def __init__(self) -> None:
        self.alert_history = deque(maxlen=1000)
        self.alert_rules = {
            "critical": [
                "supplier_performance",
                "capacity_constraint",
                "low_inventory",
                "geopolitical_risk",
            ],
            "warning": ["quality_issue", "market_downturn"],
            "info": ["system_performance"],
        }

    def process_alert(self, alert: Dict[str, Any]) -> Dict[str, Any]:
        processed_alert = dict(alert)
        processed_alert["processed_at"] = datetime.now()
        processed_alert["acknowledged"] = False

        for category, alert_types in self.alert_rules.items():
            if processed_alert["type"] in alert_types:
                processed_alert["category"] = category
                break
        else:
            processed_alert["category"] = "info"

        self.alert_history.append(processed_alert)
        return processed_alert

    def get_alert_statistics(self) -> Dict[str, Any]:
        stats = {
            "total_alerts": len(self.alert_history),
            "by_category": defaultdict(int),
            "by_type": defaultdict(int),
            "by_severity": defaultdict(int),
        }
        for alert in self.alert_history:
            stats["by_category"][alert["category"]] += 1
            stats["by_type"][alert["type"]] += 1
            stats["by_severity"][alert["severity"]] += 1
        return stats

## 3. Build the simulation

`DigitalTwinSimulation` owns the simulated state and coordinates each time step. The sequence inside `simulate_real_time_data` matters: sensor observations are generated first, then supplier, market, and demand records are updated; predictions and sourcing decisions use those records; alerts and system metrics are collected afterward.

The score used to rank a supplier is

$$
S_j = 0.4(1 - R_j) + 0.3\left(\frac{1}{C_j}\right) + 0.3Q_j,
$$

where $S_j$ is supplier $j$'s composite score, $R_j$ is its predicted risk score, $C_j$ is its predicted unit cost, and $Q_j$ is its recorded reliability score. Higher scores rank first. The reciprocal cost term rewards lower predicted cost, while the risk and reliability terms reward lower risk and higher reliability. These weights and the criticality threshold are rules in this example, not values estimated from data.

For a component with criticality greater than 0.8 and at least two available suppliers, the two highest-ranked suppliers are selected with fixed allocation ratios of 0.7 and 0.3. Otherwise, the top-ranked supplier receives the entire allocation. No capacity or eligibility constraints are applied.

In [ ]:
class DigitalTwinSimulation:
    """Coordinate a synthetic, minute-stepped sourcing simulation."""

    def __init__(self, components: List[Dict[str, Any]], suppliers: List[Dict[str, Any]]):
        self.components = components
        self.suppliers = suppliers
        self.current_time = datetime.now()
        self.iot_data_stream = deque(maxlen=10000)
        self.supplier_performance = defaultdict(list)
        self.market_conditions = deque(maxlen=1000)
        self.component_demands = defaultdict(list)
        self.sourcing_decisions = deque(maxlen=1000)
        self.dt_metrics = deque(maxlen=1000)

        self.iot_sensors = self._initialize_iot_sensors()
        self.predictive_models = self._initialize_predictive_models()
        self.scenario_engine = ScenarioEngine()
        self.alert_system = AlertSystem()

        self.start_time = datetime.now()
        self.total_data_points = 0
        self.total_decisions = 0
        self.total_alerts = 0

    def _initialize_iot_sensors(self) -> Dict[str, Dict[str, Any]]:
        sensors = {}
        for supplier in self.suppliers:
            supplier_id = supplier["id"]
            for metric, sensor_type, rate, unit in [
                ("delivery", "delivery_performance", 1.0, "fraction"),
                ("quality", "quality_score", 0.1, "score"),
                ("capacity", "capacity_utilization", 0.5, "fraction"),
            ]:
                sensors[f"supplier_{supplier_id}_{metric}"] = {
                    "type": sensor_type,
                    "location": supplier["location"],
                    "sampling_rate": rate,
                    "unit": unit,
                }

        for component in self.components:
            component_id = component["id"]
            sensors[f"component_{component_id}_inventory"] = {
                "type": "inventory_level",
                "location": "warehouse",
                "sampling_rate": 2.0,
                "unit": "units",
            }
            sensors[f"component_{component_id}_demand"] = {
                "type": "demand_rate",
                "location": "warehouse",
                "sampling_rate": 1.0,
                "unit": "units_per_hour",
            }

        sensors["market_index"] = {
            "type": "market_condition",
            "location": "global",
            "sampling_rate": 0.1,
            "unit": "index",
        }
        sensors["commodity_prices"] = {
            "type": "commodity_price",
            "location": "global",
            "sampling_rate": 0.05,
            "unit": "index",
        }
        return sensors

    def _initialize_predictive_models(self) -> Dict[str, Any]:
        return {
            "demand_forecast": DemandForecastModel(),
            "supplier_risk": SupplierRiskModel(),
            "cost_prediction": CostPredictionModel(),
            "disruption_prediction": DisruptionPredictionModel(),
        }

    def simulate_real_time_data(self, duration_minutes: int = 60) -> None:
        """Advance the synthetic system by the requested number of minutes."""
        print(f"Starting simulated data stream for {duration_minutes} minutes...")

        for minute in range(duration_minutes):
            self.current_time += timedelta(minutes=1)
            iot_data = self._generate_iot_data()
            self.iot_data_stream.extend(iot_data)
            self.total_data_points += len(iot_data)

            for performance in self._update_supplier_performance():
                self.supplier_performance[performance.supplier_id].append(performance)

            self.market_conditions.append(self._update_market_conditions())
            for demand in self._update_component_demands():
                self.component_demands[demand.component_id].append(demand)

            predictions = self._run_predictive_analytics()
            decisions = self._generate_sourcing_decisions(predictions)
            for decision in decisions:
                self.sourcing_decisions.append(decision)
                self.total_decisions += 1

            alerts = self._check_alerts()
            for alert in alerts:
                self.alert_system.process_alert(alert)
            self.total_alerts += len(alerts)

            self.dt_metrics.append(self._calculate_dt_metrics(len(alerts), len(decisions)))

            if minute % 10 == 0:
                print(
                    f"  Simulated minute {minute + 1}: {len(iot_data)} observations, "
                    f"{len(decisions)} recommendations, {len(alerts)} alerts"
                )

        print(f"Simulation complete: {self.total_data_points} observations processed.")

    def _generate_iot_data(self) -> List[IoTDataPoint]:
        data_points = []
        for sensor_id, config in self.iot_sensors.items():
            sensor_type = config["type"]
            if sensor_type == "delivery_performance":
                value = np.clip(0.85 + np.random.normal(0.0, 0.1), 0.5, 1.0)
            elif sensor_type == "quality_score":
                value = np.clip(0.90 + np.random.normal(0.0, 0.05), 0.7, 1.0)
            elif sensor_type == "capacity_utilization":
                value = np.clip(0.75 + np.random.normal(0.0, 0.15), 0.3, 1.0)
            elif sensor_type == "inventory_level":
                value = max(0.0, np.random.normal(500.0, 100.0))
            elif sensor_type == "demand_rate":
                value = max(0.0, np.random.normal(20.0, 5.0))
            elif sensor_type == "market_condition":
                value = np.random.normal(0.0, 0.3)
            elif sensor_type == "commodity_price":
                value = max(0.0, np.random.normal(100.0, 10.0))
            else:
                value = np.random.normal(0.0, 1.0)

            quality_score = float(np.clip(0.9 + np.random.normal(0.0, 0.1), 0.0, 1.0))
            data_points.append(
                IoTDataPoint(
                    sensor_id=sensor_id,
                    timestamp=self.current_time,
                    value=float(value),
                    unit=config["unit"],
                    location=config["location"],
                    quality_score=quality_score,
                )
            )
        return data_points

    def _current_sensor_values(self, sensor_ids: List[str]) -> Dict[str, float]:
        """Read this time step's observations rather than averaging old history."""
        return {
            point.sensor_id: point.value
            for point in self.iot_data_stream
            if point.timestamp == self.current_time and point.sensor_id in sensor_ids
        }

    def _update_supplier_performance(self) -> List[SupplierPerformance]:
        performances = []
        for supplier in self.suppliers:
            supplier_id = supplier["id"]
            sensor_ids = [
                f"supplier_{supplier_id}_delivery",
                f"supplier_{supplier_id}_quality",
                f"supplier_{supplier_id}_capacity",
            ]
            values = self._current_sensor_values(sensor_ids)
            performance = SupplierPerformance(
                supplier_id=supplier_id,
                timestamp=self.current_time,
                on_time_delivery_rate=values.get(sensor_ids[0], 0.85),
                quality_score=values.get(sensor_ids[1], 0.90),
                cost_performance=supplier["unit_cost"],
                inventory_turnover=max(0.0, float(np.random.normal(8.0, 2.0))),
                disruption_risk_score=max(0.0, 1.0 - values.get(sensor_ids[0], 0.85)),
                capacity_utilization=values.get(sensor_ids[2], 0.75),
                response_time=max(0.0, float(np.random.normal(24.0, 6.0))),
            )
            performances.append(performance)
        return performances

    def _update_market_conditions(self) -> MarketCondition:
        values = self._current_sensor_values(["market_index", "commodity_prices"])
        return MarketCondition(
            timestamp=self.current_time,
            market_index=values.get("market_index", 0.0),
            commodity_price_index=values.get("commodity_prices", 100.0),
            shipping_cost_index=max(0.0, 100.0 + np.random.normal(0.0, 15.0)),
            currency_exchange_rate=max(0.0, 1.0 + np.random.normal(0.0, 0.05)),
            geopolitical_risk_score=float(np.clip(np.random.normal(0.2, 0.1), 0.0, 1.0)),
            demand_volatility=max(0.0, 0.3 + np.random.normal(0.0, 0.1)),
        )

    def _update_component_demands(self) -> List[ComponentDemand]:
        demands = []
        for component in self.components:
            sensor_id = f"component_{component['id']}_demand"
            values = self._current_sensor_values([sensor_id])
            current_demand = values.get(sensor_id, 20.0)
            forecast_model = self.predictive_models["demand_forecast"]
            demands.append(
                ComponentDemand(
                    component_id=component["id"],
                    timestamp=self.current_time,
                    current_demand=current_demand,
                    forecast_7_days=forecast_model.predict(component["id"], current_demand, 7),
                    forecast_30_days=forecast_model.predict(component["id"], current_demand, 30),
                    forecast_confidence=float(np.clip(0.85 + np.random.normal(0.0, 0.1), 0.0, 1.0)),
                    safety_stock_level=component["safety_stock"],
                    reorder_point=component["reorder_point"],
                )
            )
        return demands

    def _run_predictive_analytics(self) -> Dict[str, Any]:
        predictions = {}
        recent_market = list(self.market_conditions)[-5:]

        for supplier in self.suppliers:
            supplier_id = supplier["id"]
            performances = self.supplier_performance[supplier_id]
            if performances:
                predictions[f"supplier_{supplier_id}_risk"] = (
                    self.predictive_models["supplier_risk"].predict(
                        performances[-1], recent_market
                    )
                )
            predictions[f"supplier_{supplier_id}_cost"] = (
                self.predictive_models["cost_prediction"].predict(
                    supplier_id, supplier["unit_cost"], recent_market
                )
            )

        predictions["disruption_risk"] = self.predictive_models[
            "disruption_prediction"
        ].predict(recent_market, self.suppliers)
        return predictions

    def _generate_sourcing_decisions(
        self, predictions: Dict[str, Any]
    ) -> List[SourcingDecision]:
        decisions = []
        for component in self.components:
            if not self.component_demands[component["id"]]:
                continue

            supplier_scores = []
            for supplier in self.suppliers:
                supplier_id = supplier["id"]
                risk_score = predictions.get(f"supplier_{supplier_id}_risk", 0.2)
                cost_prediction = predictions.get(
                    f"supplier_{supplier_id}_cost", supplier["unit_cost"]
                )
                if cost_prediction <= 0.0:
                    raise ValueError("Predicted supplier costs must be positive to calculate scores.")

                score = (
                    (1.0 - risk_score) * 0.4
                    + (1.0 / cost_prediction) * 0.3
                    + supplier["reliability_score"] * 0.3
                )
                supplier_scores.append((supplier_id, score, risk_score))

            supplier_scores.sort(key=lambda item: item[1], reverse=True)
            is_critical = component["criticality_score"] > 0.8

            if is_critical and len(supplier_scores) >= 2:
                selected_suppliers = [supplier_scores[0][0], supplier_scores[1][0]]
                allocation_ratios = [0.7, 0.3]
                action_type = "dual_sourcing"
                cost_impact = 0.05
                service_impact = 0.15
                risk_impact = -0.4
                confidence = 0.85
                implementation_time = timedelta(days=14)
            else:
                selected_suppliers = [supplier_scores[0][0]]
                allocation_ratios = [1.0]
                action_type = "single_sourcing"
                cost_impact = 0.0
                service_impact = 0.0
                risk_impact = 0.0
                confidence = 0.90
                implementation_time = timedelta(days=7)

            if not np.isclose(sum(allocation_ratios), 1.0):
                raise AssertionError("Sourcing allocation ratios must sum to one.")

            decisions.append(
                SourcingDecision(
                    decision_id=f"decision_{self.total_decisions + len(decisions)}",
                    timestamp=self.current_time,
                    component_id=component["id"],
                    action_type=action_type,
                    selected_suppliers=selected_suppliers,
                    allocation_ratios=allocation_ratios,
                    expected_cost_impact=cost_impact,
                    expected_service_impact=service_impact,
                    expected_risk_impact=risk_impact,
                    confidence_score=confidence,
                    implementation_time=implementation_time,
                )
            )
        return decisions

    def _check_alerts(self) -> List[Dict[str, Any]]:
        alerts = []
        for supplier_id, performances in self.supplier_performance.items():
            if not performances or performances[-1].timestamp != self.current_time:
                continue
            performance = performances[-1]
            if performance.on_time_delivery_rate < 0.8:
                alerts.append({
                    "type": "supplier_performance",
                    "severity": "high",
                    "message": f"Supplier {supplier_id} delivery rate is below 80%.",
                    "timestamp": self.current_time,
                    "value": performance.on_time_delivery_rate,
                })
            if performance.quality_score < 0.85:
                alerts.append({
                    "type": "quality_issue",
                    "severity": "medium",
                    "message": f"Supplier {supplier_id} quality score is below 85%.",
                    "timestamp": self.current_time,
                    "value": performance.quality_score,
                })
            if performance.capacity_utilization > 0.9:
                alerts.append({
                    "type": "capacity_constraint",
                    "severity": "high",
                    "message": f"Supplier {supplier_id} capacity utilization is above 90%.",
                    "timestamp": self.current_time,
                    "value": performance.capacity_utilization,
                })

        for component in self.components:
            component_id = component["id"]
            inventory_id = f"component_{component_id}_inventory"
            inventory = self._current_sensor_values([inventory_id]).get(inventory_id)
            if inventory is not None and inventory < component["reorder_point"]:
                alerts.append({
                    "type": "low_inventory",
                    "severity": "high",
                    "message": f"Component {component_id} inventory is below its reorder point.",
                    "timestamp": self.current_time,
                    "value": inventory,
                })

        if self.market_conditions:
            market = self.market_conditions[-1]
            if market.market_index < -0.5:
                alerts.append({
                    "type": "market_downturn",
                    "severity": "medium",
                    "message": "The market indicator is below the downturn threshold.",
                    "timestamp": self.current_time,
                    "value": market.market_index,
                })
            if market.geopolitical_risk_score > 0.7:
                alerts.append({
                    "type": "geopolitical_risk",
                    "severity": "high",
                    "message": "The simulated geopolitical risk score is above 0.7.",
                    "timestamp": self.current_time,
                    "value": market.geopolitical_risk_score,
                })
        return alerts

    def _calculate_dt_metrics(
        self, alerts_generated: int, decisions_supported: int
    ) -> DigitalTwinMetrics:
        recent_data = list(self.iot_data_stream)[-100:]
        data_quality = (
            float(np.mean([point.quality_score for point in recent_data]))
            if recent_data
            else 0.9
        )
        prediction_accuracy = float(np.clip(0.85 + np.random.normal(0.0, 0.05), 0.7, 0.95))
        uptime = float(np.clip(0.995 + np.random.normal(0.0, 0.005), 0.98, 1.0))
        response_time = max(10.0, 50.0 + np.random.normal(0.0, 20.0))
        sync_delay = max(50.0, 100.0 + np.random.normal(0.0, 50.0))

        return DigitalTwinMetrics(
            timestamp=self.current_time,
            data_quality_score=data_quality,
            prediction_accuracy=prediction_accuracy,
            system_uptime=uptime,
            response_time_ms=float(response_time),
            synchronization_delay_ms=float(sync_delay),
            active_sensors=len(self.iot_sensors),
            data_points_processed=len(recent_data),
            alerts_generated=alerts_generated,
            decisions_supported=decisions_supported,
        )

    def run_scenario_analysis(
        self,
        scenario_name: str,
        scenario_params: Dict[str, Any],
        duration_minutes: int = 30,
    ) -> Dict[str, Any]:
        """Run a scenario from a fresh history, then restore the base state."""
        print(f"Running scenario: {scenario_name}")
        original_state = copy.deepcopy(self.__dict__)
        try:
            self._apply_scenario_params(scenario_params)
            self._reset_simulation_history()
            self.simulate_real_time_data(duration_minutes=duration_minutes)
            return self._collect_scenario_results()
        finally:
            self.__dict__.clear()
            self.__dict__.update(original_state)

    def _reset_simulation_history(self) -> None:
        self.current_time = datetime.now()
        self.iot_data_stream = deque(maxlen=10000)
        self.supplier_performance = defaultdict(list)
        self.market_conditions = deque(maxlen=1000)
        self.component_demands = defaultdict(list)
        self.sourcing_decisions = deque(maxlen=1000)
        self.dt_metrics = deque(maxlen=1000)
        self.alert_system = AlertSystem()
        self.start_time = datetime.now()
        self.total_data_points = 0
        self.total_decisions = 0
        self.total_alerts = 0

    def _apply_scenario_params(self, params: Dict[str, Any]) -> None:
        if "supplier_disruption" in params:
            disrupted_supplier = params["supplier_disruption"]
            for supplier in self.suppliers:
                if supplier["id"] == disrupted_supplier:
                    supplier["reliability_score"] *= 0.5
                    supplier["unit_cost"] *= 1.3

        if "demand_surge" in params:
            surge_factor = params["demand_surge"]
            for component in self.components:
                component["demand_forecast"] *= surge_factor

        if "cost_increase" in params:
            cost_factor = params["cost_increase"]
            for supplier in self.suppliers:
                supplier["unit_cost"] *= cost_factor

    def _collect_scenario_results(self) -> Dict[str, Any]:
        decision_types = defaultdict(int)
        for decision in self.sourcing_decisions:
            decision_types[decision.action_type] += 1

        alert_types = defaultdict(int)
        for alert in self.alert_system.alert_history:
            alert_types[alert["type"]] += 1

        return {
            "total_decisions": self.total_decisions,
            "total_alerts": self.total_alerts,
            "avg_prediction_accuracy": (
                float(np.mean([metric.prediction_accuracy for metric in self.dt_metrics]))
                if self.dt_metrics else 0.0
            ),
            "avg_data_quality": (
                float(np.mean([metric.data_quality_score for metric in self.dt_metrics]))
                if self.dt_metrics else 0.0
            ),
            "decision_types": decision_types,
            "alert_types": alert_types,
        }

    def generate_comprehensive_report(self) -> str:
        """Create a report from the state currently held by the simulation."""
        report = [
            "# Synthetic Digital-Twin Simulation Report",
            "",
            f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"Elapsed wall-clock time: {(datetime.now() - self.start_time).total_seconds():.1f} seconds",
            "",
            "## System overview",
            f"- Components: {len(self.components)}",
            f"- Suppliers: {len(self.suppliers)}",
            f"- Configured sensors: {len(self.iot_sensors)}",
            f"- Simulated observations processed: {self.total_data_points:,}",
            f"- Sourcing recommendations: {self.total_decisions}",
            f"- Alerts generated: {self.total_alerts}",
        ]

        if self.dt_metrics:
            latest = self.dt_metrics[-1]
            report.extend([
                "",
                "## Latest simulated metrics",
                f"- Data quality score: {latest.data_quality_score:.1%}",
                f"- Illustrative prediction-accuracy metric: {latest.prediction_accuracy:.1%}",
                f"- Simulated system uptime: {latest.system_uptime:.1%}",
                f"- Simulated response time: {latest.response_time_ms:.1f} ms",
                f"- Simulated synchronization delay: {latest.synchronization_delay_ms:.1f} ms",
            ])

        report.extend(["", "## Latest supplier performance"])
        for supplier_id, performances in self.supplier_performance.items():
            if performances:
                latest = performances[-1]
                report.extend([
                    f"- {supplier_id}: on-time delivery {latest.on_time_delivery_rate:.1%}; "
                    f"quality {latest.quality_score:.1%}; "
                    f"capacity utilization {latest.capacity_utilization:.1%}; "
                    f"illustrative disruption risk {latest.disruption_risk_score:.1%}",
                ])

        report.extend(["", "## Recent recommendations"])
        for decision in list(self.sourcing_decisions)[-5:]:
            report.append(
                f"- {decision.component_id}: {decision.action_type}; "
                f"suppliers {', '.join(decision.selected_suppliers)}; "
                f"allocation {decision.allocation_ratios}"
            )

        report.extend([
            "",
            "These results are generated from synthetic inputs and illustrative rules. "
            "They are not live operational measurements or validated business forecasts.",
        ])
        return "\n".join(report)

## 4. Create the example system

The records below define the example inputs. The component and supplier values are simulation inputs, not observations from live operations. The architecture labels describe the conceptual organization used here: physical entities, simulated connectivity, data processing, and decision-support outputs.

In [ ]:
def create_digital_twin_example() -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Create the component and supplier inputs for the MediTech example."""
    components_data = [
        {
            "id": "comp_001",
            "name": "Component_1_Critical",
            "criticality_score": 0.9,
            "current_inventory": 230.0,
            "safety_stock": 46.0,
            "reorder_point": 69.0,
            "demand_forecast": 11.5,
        },
        {
            "id": "comp_002",
            "name": "Component_2_Standard",
            "criticality_score": 0.6,
            "current_inventory": 405.0,
            "safety_stock": 81.0,
            "reorder_point": 121.5,
            "demand_forecast": 20.25,
        },
        {
            "id": "comp_003",
            "name": "Component_3_Medium",
            "criticality_score": 0.75,
            "current_inventory": 160.0,
            "safety_stock": 32.0,
            "reorder_point": 48.0,
            "demand_forecast": 6.15,
        },
    ]

    suppliers_data = [
        {
            "id": "supplier_1",
            "name": "Supplier_A",
            "location": "Asia",
            "unit_cost": 15.0,
            "reliability_score": 0.85,
            "quality_score": 0.87,
        },
        {
            "id": "supplier_2",
            "name": "Supplier_B",
            "location": "Europe",
            "unit_cost": 18.0,
            "reliability_score": 0.92,
            "quality_score": 0.94,
        },
        {
            "id": "supplier_3",
            "name": "Supplier_C",
            "location": "North America",
            "unit_cost": 16.0,
            "reliability_score": 0.88,
            "quality_score": 0.90,
        },
        {
            "id": "supplier_4",
            "name": "Supplier_D",
            "location": "Asia",
            "unit_cost": 14.0,
            "reliability_score": 0.82,
            "quality_score": 0.85,
        },
    ]
    return components_data, suppliers_data


components_data, suppliers_data = create_digital_twin_example()
digital_twin = DigitalTwinSimulation(components_data, suppliers_data)

print("Synthetic digital-twin simulation initialized:")
print(f"  Components: {len(components_data)}")
print(f"  Suppliers: {len(suppliers_data)}")
print(f"  Configured sensors: {len(digital_twin.iot_sensors)}")
print(f"  Illustrative prediction models: {len(digital_twin.predictive_models)}")
print("\nConceptual architecture:")
print("  Layer 1 (Physical): Components, suppliers, and operations")
print("  Layer 2 (Connectivity): Simulated sensors and data stream")
print("  Layer 3 (Data): Processing, analytics, and stored histories")
print("  Layer 4 (Application): Recommendations, scenarios, and alerts")

## 5. Run the base simulation

Each loop iteration represents one simulated minute. The console messages report progress only; they do not imply that the notebook is connected to a live system. The random inputs mean two runs can produce different histories and recommendations.

In [ ]:
print("=== SYNTHETIC SIMULATION ===")
digital_twin.simulate_real_time_data(duration_minutes=60)

print("\n=== BASE-RUN SUMMARY ===")
print(f"Simulated observations processed: {digital_twin.total_data_points:,}")
print(f"Sourcing recommendations generated: {digital_twin.total_decisions}")
print(f"Alerts generated: {digital_twin.total_alerts}")

if digital_twin.dt_metrics:
    latest_metrics = digital_twin.dt_metrics[-1]
    print("\nLatest simulated metrics:")
    print(f"  Data quality: {latest_metrics.data_quality_score:.1%}")
    print(f"  Illustrative prediction-accuracy metric: {latest_metrics.prediction_accuracy:.1%}")
    print(f"  Simulated uptime: {latest_metrics.system_uptime:.1%}")
    print(f"  Simulated response time: {latest_metrics.response_time_ms:.1f} ms")
    print(f"  Simulated synchronization delay: {latest_metrics.synchronization_delay_ms:.1f} ms")

## 6. Compare what-if scenarios

A scenario applies specified changes to supplier reliability and cost or to component demand-forecast inputs, then runs a fresh simulated history. The base simulation state is restored after each scenario, so scenario runs do not accumulate changes in the base data. Because random inputs are unseeded, scenario results are not controlled experiments: differences can reflect both the scenario parameters and random variation.

In [ ]:
scenarios = digital_twin.scenario_engine.get_available_scenarios()
print("Available scenarios:")
for scenario_id, scenario_info in scenarios.items():
    print(f"  {scenario_id}: {scenario_info['description']}")

scenario_results = {}
for scenario_id, scenario_info in scenarios.items():
    print(f"\n--- {scenario_id} ---")
    results = digital_twin.run_scenario_analysis(
        scenario_name=scenario_id,
        scenario_params=scenario_info["params"],
        duration_minutes=scenario_info["duration_minutes"],
    )
    scenario_results[scenario_id] = results
    print(f"  Recommendations: {results['total_decisions']}")
    print(f"  Alerts: {results['total_alerts']}")
    print(f"  Mean illustrative prediction-accuracy metric: {results['avg_prediction_accuracy']:.1%}")
    print(f"  Mean data-quality score: {results['avg_data_quality']:.1%}")
    print(f"  Recommendation types: {dict(results['decision_types'])}")
    print(f"  Alert types: {dict(results['alert_types'])}")

## 7. Visualize simulation histories and scenario summaries

The first four panels describe the base run. The scenario panel compares counts from the independent scenario runs. The final panel summarizes selected average base-run metrics; response time and synchronization delay are transformed into bounded illustrative scores for display, so those two bars are not measurements on the same scale as data quality or prediction accuracy.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Synthetic Sourcing Simulation: Monitoring and Scenario Summaries", fontsize=15)

if digital_twin.dt_metrics:
    timestamps = [metric.timestamp for metric in digital_twin.dt_metrics]
    minutes = [(timestamp - timestamps[0]).total_seconds() / 60.0 for timestamp in timestamps]
    data_quality = [metric.data_quality_score for metric in digital_twin.dt_metrics]
    prediction_accuracy = [metric.prediction_accuracy for metric in digital_twin.dt_metrics]
    response_times = [metric.response_time_ms for metric in digital_twin.dt_metrics]
    sync_delays = [metric.synchronization_delay_ms for metric in digital_twin.dt_metrics]
    decisions_supported = [metric.decisions_supported for metric in digital_twin.dt_metrics]
    alerts_generated = [metric.alerts_generated for metric in digital_twin.dt_metrics]

    ax = axes[0, 0]
    ax.plot(minutes, data_quality, linewidth=2, label="Data quality")
    ax.plot(minutes, prediction_accuracy, linewidth=2, label="Illustrative prediction metric")
    ax.set_xlabel("Simulated time (minutes)")
    ax.set_ylabel("Score (0–1)")
    ax.set_title("Data and Prediction Metrics")
    ax.set_ylim(0.0, 1.0)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(minutes, response_times, linewidth=2, label="Response time")
    ax.plot(minutes, sync_delays, linewidth=2, label="Synchronization delay")
    ax.set_xlabel("Simulated time (minutes)")
    ax.set_ylabel("Simulated time (ms)")
    ax.set_title("Illustrative System Timings")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 2]
    ax.bar(minutes, decisions_supported, alpha=0.7, label="Recommendations")
    ax.bar(minutes, alerts_generated, alpha=0.7, label="Alerts")
    ax.set_xlabel("Simulated time (minutes)")
    ax.set_ylabel("Count per time step")
    ax.set_title("Recommendation and Alert Activity")
    ax.legend()
    ax.grid(True, alpha=0.3)

    supplier_names = []
    delivery_rates = []
    quality_scores = []
    for supplier_id, performances in digital_twin.supplier_performance.items():
        if performances:
            supplier_names.append(supplier_id)
            delivery_rates.append(performances[-1].on_time_delivery_rate)
            quality_scores.append(performances[-1].quality_score)

    ax = axes[1, 0]
    positions = np.arange(len(supplier_names))
    width = 0.35
    ax.bar(positions - width / 2, delivery_rates, width, label="On-time delivery")
    ax.bar(positions + width / 2, quality_scores, width, label="Quality score")
    ax.set_xlabel("Supplier")
    ax.set_ylabel("Score (0–1)")
    ax.set_title("Latest Supplier Metrics")
    ax.set_xticks(positions)
    ax.set_xticklabels(supplier_names, rotation=35, ha="right")
    ax.set_ylim(0.0, 1.0)
    ax.legend()
    ax.grid(True, alpha=0.3)

scenario_names = list(scenario_results)
scenario_decisions = [scenario_results[name]["total_decisions"] for name in scenario_names]
scenario_alerts = [scenario_results[name]["total_alerts"] for name in scenario_names]
ax = axes[1, 1]
positions = np.arange(len(scenario_names))
width = 0.35
ax.bar(positions - width / 2, scenario_decisions, width, label="Recommendations")
ax.bar(positions + width / 2, scenario_alerts, width, label="Alerts")
ax.set_xlabel("Scenario")
ax.set_ylabel("Count")
ax.set_title("Independent Scenario Run Counts")
ax.set_xticks(positions)
ax.set_xticklabels([name.replace("_", "\n") for name in scenario_names], rotation=0)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
if digital_twin.dt_metrics:
    mean_data_quality = float(np.mean(data_quality))
    mean_prediction_accuracy = float(np.mean(prediction_accuracy))
    mean_response = float(np.mean(response_times))
    mean_sync = float(np.mean(sync_delays))
    normalized_response = float(np.clip(1.0 - max(0.0, mean_response - 50.0) / 100.0, 0.0, 1.0))
    normalized_sync = float(np.clip(1.0 - max(0.0, mean_sync - 100.0) / 200.0, 0.0, 1.0))
    health_metrics = {
        "Data quality": mean_data_quality,
        "Prediction metric": mean_prediction_accuracy,
        "Response-time score": normalized_response,
        "Sync-delay score": normalized_sync,
    }
    bars = ax.bar(list(health_metrics), list(health_metrics.values()))
    ax.set_ylabel("Score (0–1; timing scores normalized)")
    ax.set_title("Selected Base-Run Summary Metrics")
    ax.set_ylim(0.0, 1.0)
    ax.tick_params(axis="x", rotation=25)
    ax.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, health_metrics.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.2f}", ha="center")
else:
    ax.set_title("No base-run metrics available")

plt.tight_layout()
plt.show()

## 8. Read the report and interpret the results

The report summarizes the state held by the base simulation. Scenario analysis temporarily uses a separate state and restores the base state afterward. Consequently, the report describes the base run, not the last scenario. The metrics called prediction accuracy, uptime, response time, and synchronization delay are simulated indicators in this notebook; they are not measured production-system performance.

In [ ]:
print(digital_twin.generate_comprehensive_report())

print("\n=== IMPLEMENTATION SUMMARY ===")
print("  Simulated sensor observations, supplier records, and market indicators")
print("  Simple illustrative forecasting and risk/cost scoring models")
print("  Rule-based single- or dual-sourcing recommendations")
print("  Scenario runs, alert records, and summary visualizations")
print("\nInterpretation:")
print("  Treat the outputs as demonstrations of a simulation architecture.")
print("  They do not establish cost savings, predictive accuracy, or production readiness.")

## 9. Strengths, limitations, and extensions

### What this tier demonstrates

- **System integration as a design pattern:** sensor-like observations, data processing, models, decision rules, alerts, and reports are connected in one workflow.
- **Predictive inputs to decision support:** the supplier ranking uses estimated risk and cost alongside recorded reliability.
- **What-if analysis:** scenario parameters can represent supplier disruption, demand pressure, cost increases, or combinations of these conditions.
- **Operational visibility:** histories and plots expose how simulated metrics vary during a run.

### Limitations

- **Synthetic data:** simulated observations do not establish real-world performance or data quality.
- **Uncalibrated models:** model coefficients, noise distributions, and illustrative performance metrics are not learned from historical data.
- **Rule-based sourcing:** the ranking is not an optimization model. It does not minimize total procurement cost or enforce capacity, eligibility, service, or contractual constraints.
- **Simplified scenarios:** scenario inputs alter selected records but do not model causal operational dynamics such as production delays or inventory depletion.
- **No live integration:** no sensor, ERP, market feed, or supplier system is connected.
- **No reproducibility guarantee across runs:** no random seed is set, so random draws can change results.

### Possible extensions within this simulation framework

A larger study could use validated operational data, document calibration and evaluation procedures, add constraints that reflect the actual sourcing problem, and test scenarios with controlled assumptions. Those extensions require problem-specific data and assumptions; they are not implemented here.

### Reading the results responsibly

A recommendation is the output of the stated scoring rule and criticality threshold. It is not evidence that a particular supplier choice is globally optimal. Scenario differences are exploratory because the runs use random inputs and are not controlled experiments. Any business conclusion would require appropriate data, validation, and constraints that represent the real sourcing decision.